[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-04-24-direct-logit-attribution/challenge.ipynb)

# Direct Logit Attribution — Decomposing Transformer Predictions

**AI Safety Challenge · Day 10 · 2026-04-24 · short-coding · ~25 min**

> **Relevance:** Direct Logit Attribution (DLA) is the workhorse for identifying *which* attention
> heads and MLP layers drive a model's prediction. Every circuit paper — IOI, indirect object
> identification, refusal direction — uses DLA or a close variant as its first diagnostic tool.
> Implementing DLA from scratch makes the linearity argument concrete: the residual stream's additive
> structure is what allows mechanistic interpretability at all. This is core LASR/SAIGE preparation.

## Background

In a residual-stream transformer (no LayerNorm for now — we fold it in or assume it's approximately
linear), the logit for any output token `t` at the final position is:

```
logit(t) ≈ W_U[t] @ x_final
```

where `x_final` is the residual stream at the last token, and `W_U[t]` is the row of the unembedding
matrix corresponding to token `t`. Because the residual stream accumulates contributions additively:

```
x_final = embed + Σ_layers (attn_out_L + mlp_out_L)
```

we can distribute the dot product:

```
logit(t) ≈ Σ_c (W_U[t] @ component_c)
```

Each component gets a **scalar credit**: positive → pushes model toward predicting `t`, negative →
pushes away. On the IOI task ("Mary and John went to the store, John gave the bag to ___"), the
**logit difference** (IO token − S token) is the key metric, and DLA tells us exactly which heads
are name-movers vs. S-inhibitors.


## Setup


In [ ]:
import numpy as np

np.random.seed(42)

# Synthetic residual-stream components (d_model=4 for hand-checkable math)
d_model = 4

component_outputs = {
    "embed":  np.array([1.0, 0.0,  0.0, 0.0]),   # token embedding
    "L0H0":   np.array([0.0, 2.0,  0.0, 0.0]),   # 'Name Mover' head
    "L0H1":   np.array([0.0, 0.0, -1.0, 0.0]),   # 'S-Inhibition' head
    "L1_mlp": np.array([0.0, 0.0,  0.0, 0.5]),   # MLP layer 1
}

# Unembedding rows for the IO token ('Mary') and S token ('John')
W_U_IO = np.array([0.0, 1.0,  0.0, 0.0])   # direction that reads out 'Mary'
W_U_S  = np.array([0.0, 0.0,  1.0, 0.0])   # direction that reads out 'John'

# Ground-truth total logit difference (you can verify by hand)
x_final = sum(component_outputs.values())
TRUE_LOGIT_DIFF = float(W_U_IO @ x_final - W_U_S @ x_final)  # should be 3.0
print(f'Total logit diff: {TRUE_LOGIT_DIFF}')  # 3.0


## Task 1 — `token_logit_contribution`

Compute a single component's contribution to **one** token's logit.
This is the atomic building block of DLA.

```
contribution_to_t = W_U[t] · component_output
```

**Args:**
- `component_output`: shape `(d_model,)` — the component's residual-stream contribution
- `W_U_row`: shape `(d_model,)` — the unembedding row for token `t`

**Returns:** `float`


In [ ]:
def token_logit_contribution(component_output: np.ndarray, W_U_row: np.ndarray) -> float:
    """
    Compute component_output's contribution to the logit for the token
    whose unembedding row is W_U_row.
    """
    raise NotImplementedError


### Tests — Task 1


In [ ]:
# L0H0 = [0,2,0,0], W_U_IO = [0,1,0,0]  →  dot product = 2.0
assert token_logit_contribution(component_outputs['L0H0'], W_U_IO) == 2.0, \
    'L0H0 contribution to IO logit should be 2.0'

# L0H1 = [0,0,-1,0], W_U_IO = [0,1,0,0]  →  dot product = 0.0
assert token_logit_contribution(component_outputs['L0H1'], W_U_IO) == 0.0, \
    'L0H1 has no projection onto W_U_IO'

# embed = [1,0,0,0], W_U_S = [0,0,1,0]  →  dot product = 0.0
assert token_logit_contribution(component_outputs['embed'], W_U_S) == 0.0, \
    'embed has no projection onto W_U_S'

print('Task 1 tests passed!')


## Task 2 — `compute_dla`

Compute each component's contribution to the **logit difference** (IO − S).

For each component `c`:
```
DLA[c] = (W_U_IO − W_U_S) · component_outputs[c]
       = token_logit_contribution(c, W_U_IO) − token_logit_contribution(c, W_U_S)
```

**Linearity guarantee:** `Σ_c DLA[c]` should equal the total logit difference.

**Args:**
- `component_outputs`: `dict[str, np.ndarray]`
- `W_U_IO`, `W_U_S`: `np.ndarray` of shape `(d_model,)`

**Returns:** `dict[str, float]`


In [ ]:
def compute_dla(
    component_outputs: dict,
    W_U_IO: np.ndarray,
    W_U_S: np.ndarray,
) -> dict:
    """
    Return a dict mapping component name -> logit-difference attribution.
    Uses token_logit_contribution as its primitive.
    """
    raise NotImplementedError


### Tests — Task 2


In [ ]:
dla = compute_dla(component_outputs, W_U_IO, W_U_S)

# Expected values (verify by hand with d_model=4 vectors above):
#   embed:  (W_U_IO - W_U_S) @ [1,0,0,0] = [0,1,-1,0].[1,0,0,0] = 0
#   L0H0:   [0,1,-1,0].[0,2,0,0]  = 2.0
#   L0H1:   [0,1,-1,0].[0,0,-1,0] = 1.0  (S-inhibition: suppresses 'John')
#   L1_mlp: [0,1,-1,0].[0,0,0,0.5]= 0.0
assert abs(dla['embed']  - 0.0) < 1e-10, f'embed DLA should be 0, got {dla["embed"]}'
assert abs(dla['L0H0']   - 2.0) < 1e-10, f'L0H0 DLA should be 2.0, got {dla["L0H0"]}'
assert abs(dla['L0H1']   - 1.0) < 1e-10, f'L0H1 DLA should be 1.0, got {dla["L0H1"]}'
assert abs(dla['L1_mlp'] - 0.0) < 1e-10, f'L1_mlp DLA should be 0, got {dla["L1_mlp"]}'

# Linearity check: attributions must sum to total logit diff
total_dla = sum(dla.values())
assert abs(total_dla - TRUE_LOGIT_DIFF) < 1e-10, \
    f'DLA sum {total_dla} != total logit diff {TRUE_LOGIT_DIFF} — linearity violated!'

print(f'Task 2 passed! DLA: {dla}')
print(f'Sum of attributions = {total_dla} (= total logit diff {TRUE_LOGIT_DIFF})')


## Task 3 — `top_k_contributors`

Return the top-k components by **absolute DLA magnitude**, sorted descending.
This is what circuit papers show in the 'who matters most?' bar chart.

**Args:**
- `dla_dict`: `dict[str, float]` — output of `compute_dla`
- `k`: `int` — how many top contributors to return

**Returns:** `list[tuple[str, float]]` — `[(name, attribution), ...]` sorted by `|attribution|` desc


In [ ]:
def top_k_contributors(dla_dict: dict, k: int = 3) -> list:
    """
    Return the k components with the largest absolute DLA values, sorted descending.
    """
    raise NotImplementedError


### Tests — Task 3


In [ ]:
top2 = top_k_contributors(dla, k=2)

# Should be L0H0 (|2.0|) then L0H1 (|1.0|)
assert top2[0][0] == 'L0H0', f'Largest contributor should be L0H0, got {top2[0][0]}'
assert top2[1][0] == 'L0H1', f'Second contributor should be L0H1, got {top2[1][0]}'
assert len(top2) == 2, f'Expected 2 results, got {len(top2)}'

# Ordering: |first| >= |second|
assert abs(top2[0][1]) >= abs(top2[1][1]), 'Results must be sorted by |attribution| descending'

print(f'Task 3 passed! Top-2: {top2}')

# Bonus: print all components sorted for visual inspection
print('\nFull ranking:')
for name, val in top_k_contributors(dla, k=len(dla)):
    bar = '█' * int(abs(val) * 10)
    sign = '+' if val >= 0 else '-'
    print(f'  {name:12s} {sign}{abs(val):.2f}  {bar}')


## Reflection

DLA makes the residual stream's linearity *useful*: because `x_final` is a sum, `W_U @ x_final` is
also a sum — and each term is independently attributable. The moment you add non-linear interactions
(say, one head's output gates another's via multiplication), DLA breaks down and you need
path-patching or causal scrubbing to assign credit. Knowing exactly *where* the linear approximation
holds is one of the open questions in the Sharkey et al. 2025 paper below.

---

> 📚 **Further reading:** [Open Problems in Mechanistic Interpretability](https://arxiv.org/abs/2501.16496)
> — Sharkey, Chughtai, Batson, Lindsey, Wu, Bushnaq, …Nanda, McGrath (2025).
> Section on attribution methods surveys where DLA works, where it fails, and what stronger
> attribution tools are needed.


<details>
<summary><b>Solution</b></summary>

```python
def token_logit_contribution(component_output, W_U_row):
    return float(np.dot(W_U_row, component_output))


def compute_dla(component_outputs, W_U_IO, W_U_S):
    return {
        name: token_logit_contribution(out, W_U_IO) - token_logit_contribution(out, W_U_S)
        for name, out in component_outputs.items()
    }


def top_k_contributors(dla_dict, k=3):
    return sorted(dla_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:k]
```

**Key insight:** `token_logit_contribution` is just a dot product — the entire DLA machinery
collapses to one line per component once you see the residual stream as a sum.
The linearity check (`sum(DLA) == total_logit_diff`) is not just a test —
it's a theorem, and passing it proves your implementation is mathematically consistent.
When DLA fails to sum correctly in practice (e.g. with folded LayerNorm),
that's a signal that the linear approximation is breaking down for that model.

</details>
